# SOXX 2011–2018 申报索引来源清单

从 SEC 已列出的四份 iShares Trust 历史 submissions 分片取得原始字节，按接受时间列出 N-Q/N-CSR/N-CSRS 及修订候选；不读取申报正文、证券价格或 SOXX 收益。


In [1]:
import json,hashlib,subprocess,time,tarfile,tempfile,shlex
from datetime import datetime,timezone
from pathlib import Path
from src.data.nas_market_data import _settings,nas_api_call
root=Path.cwd();card=root/'research/stage1_soxx_2011_2018_filing_inventory_20261023';plan=json.loads((card/'plan.json').read_text())
def sha(p):
 h=hashlib.sha256()
 with Path(p).open('rb') as f:
  for chunk in iter(lambda:f.read(1024*1024),b''):h.update(chunk)
 return h.hexdigest()
assert plan['frozen'] and not (card/'attempt_started.json').exists()
parent_raw=root/plan['parent_source']['raw_path']
assert sha(parent_raw)==plan['parent_source']['raw_sha256']
parent=json.loads(parent_raw.read_text());assert str(parent['cik']).zfill(10)==plan['parent_source']['CIK']
assert json.loads((root/plan['parent_source']['card']/'summary.json').read_text())['source_NAS_scoped_restore_verified']
file_meta={x['name']:x for x in parent['filings']['files']}
for name in plan['sidecars_in_order']:assert name in file_meta
raw_root=root/'data/snapshots/stage1_soxx_2011_2018_filing_inventory_20261023';raw_root.mkdir(parents=True,exist_ok=True)
assert not any(raw_root.iterdir())
state={'status':'started','GET_attempts_started':0,'market_outcome_reads':0,'model_fits':0,'native_accounts':0}
(card/'attempt_started.json').write_text(json.dumps(state,indent=2)+'\n')
receipts=[];candidates=[];error=None
for j,name in enumerate(plan['sidecars_in_order']):
 state['GET_attempts_started']+=1;(card/'attempt_started.json').write_text(json.dumps(state,indent=2)+'\n')
 url=plan['base_url']+name;body=raw_root/name;headers=raw_root/(name+'.headers')
 requested=datetime.now(timezone.utc).isoformat()
 cmd=['curl','--http1.1','--noproxy','*','--connect-timeout',str(plan['request']['connect_timeout_seconds']),'--max-time',str(plan['request']['total_timeout_seconds']),'--max-filesize',str(plan['request']['max_response_bytes_each']),'--fail-with-body','-sS','-L','-D',str(headers),'-o',str(body),'-A',plan['request']['user_agent'],'-w','%{http_code} %{url_effective}',url]
 try:run=subprocess.run(cmd,capture_output=True,text=True,timeout=75)
 except Exception as exc:
  error=f'{type(exc).__name__}: {exc}';receipts.append({'name':name,'requested_at_utc':requested,'error':error});break
 received=datetime.now(timezone.utc).isoformat();fields=run.stdout.rsplit(' ',1);status=int(fields[0]) if len(fields)==2 and fields[0].isdigit() else None
 receipt={'name':name,'url':url,'requested_at_utc':requested,'received_at_utc':received,'curl_exit_code':run.returncode,'http_status':status,'final_url':fields[-1] if fields else None,'raw_sha256':sha(body) if body.exists() else None,'raw_bytes':body.stat().st_size if body.exists() else 0,'headers_sha256':sha(headers) if headers.exists() else None,'headers_bytes':headers.stat().st_size if headers.exists() else 0,'stderr':run.stderr[:300]}
 receipts.append(receipt)
 if run.returncode!=0 or status!=200 or not body.exists() or body.stat().st_size>plan['request']['max_response_bytes_each']:
  error='transport_or_size_failure';break
 try:
  source=json.loads(body.read_text());required=['accessionNumber','form','filingDate','reportDate','acceptanceDateTime','primaryDocument'];n=len(source['accessionNumber'])
  assert all(isinstance(source[k],list) and len(source[k])==n for k in required)
  actual_dates=[str(x) for x in source['filingDate']];assert all(len(x)==10 for x in actual_dates)
  receipt['total_filing_rows']=n;receipt['actual_filing_from']=min(actual_dates);receipt['actual_filing_to']=max(actual_dates)
  receipt['advertised_filing_from']=file_meta[name]['filingFrom'];receipt['advertised_filing_to']=file_meta[name]['filingTo']
  receipt['advertised_range_matches_actual']=receipt['actual_filing_from']==receipt['advertised_filing_from'] and receipt['actual_filing_to']==receipt['advertised_filing_to']
  own=[]
  for i,form in enumerate(source['form']):
   if form in plan['form_set']:
    row={'sidecar':name,'accession':source['accessionNumber'][i],'form':form,'filed':source['filingDate'][i],'report_date':source['reportDate'][i],'accepted_at':source['acceptanceDateTime'][i],'primary_document':source['primaryDocument'][i]}
    own.append(row)
  receipt['candidate_form_rows']=len(own);candidates.extend(own)
 except Exception as exc:
  error=f'schema_or_JSON_failure:{type(exc).__name__}:{exc}';receipt['parse_error']=error;break
 if j+1<len(plan['sidecars_in_order']):time.sleep(plan['request']['seconds_between_requests'])
(card/'capture_receipts.json').write_text(json.dumps(receipts,indent=2)+'\n')
(card/'candidate_inventory.json').write_text(json.dumps({'parent_raw_sha256':plan['parent_source']['raw_sha256'],'form_set':plan['form_set'],'rows':sorted(candidates,key=lambda x:(x['accepted_at'],x['accession']))},indent=2)+'\n')
manifest={'plan_sha256':sha(card/'plan.json'),'files':[{'relative_path':str(p.relative_to(raw_root)),'sha256':sha(p),'bytes':p.stat().st_size} for p in sorted(raw_root.iterdir()) if p.is_file()],'candidate_inventory_sha256':sha(card/'candidate_inventory.json'),'capture_receipts_sha256':sha(card/'capture_receipts.json')}
(card/'source_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
settings=_settings(root)
with tempfile.TemporaryDirectory(prefix='soxx-2011-2018-sidecars-') as temp:
 archive=Path(temp)/'source.tar.gz'
 with tarfile.open(archive,'w:gz') as out:
  for item in manifest['files']:out.add(raw_root/item['relative_path'],arcname='raw/'+item['relative_path'])
  for name in ['source_manifest.json','capture_receipts.json','candidate_inventory.json']:out.add(card/name,arcname=name)
 digest=sha(archive)
 evidence=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact']
 assert evidence['sha256']==digest and evidence['size_bytes']==archive.stat().st_size
 remote=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest
 with tempfile.TemporaryDirectory(prefix='soxx-sidecars-restore-') as rd:
  restored=Path(rd)/'restore.tar.gz'
  with restored.open('wb') as out:
   run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(remote)],stdout=out,stderr=subprocess.PIPE,timeout=90)
  assert run.returncode==0 and sha(restored)==digest
  with tarfile.open(restored,'r:gz') as source:
   for item in manifest['files']:
    member=source.extractfile('raw/'+item['relative_path']);assert member and hashlib.sha256(member.read()).hexdigest()==item['sha256']
status='four_sidecars_archived_candidate_inventory_only' if error is None and len(receipts)==4 else 'partial_sidecar_source_needs_repair'
result={'status':status,'error':error,'GET_attempts':state['GET_attempts_started'],'sidecars_received':sum(x.get('http_status')==200 and 'parse_error' not in x for x in receipts),'candidate_form_rows':len(candidates),'candidate_form_counts':{f:sum(x['form']==f for x in candidates) for f in plan['form_set']},'candidate_filing_first':min((x['filed'] for x in candidates),default=None),'candidate_filing_last':max((x['filed'] for x in candidates),default=None),'range_discrepancies':[x['name'] for x in receipts if x.get('advertised_range_matches_actual') is False],'nas_archive_sha256':digest,'nas_archive_bytes':evidence['size_bytes'],'nas_remote_and_scoped_restore_verified':True,'SOXX_series_schedule_qualified':False,'full_training_feature_qualified':False,'market_outcome_reads':0,'model_fits':0,'native_accounts':0}
(card/'result.json').write_text(json.dumps(result,indent=2)+'\n')
print({k:result[k] for k in ('status','GET_attempts','candidate_form_rows','candidate_form_counts','range_discrepancies','nas_archive_sha256')})


{'status': 'four_sidecars_archived_candidate_inventory_only', 'GET_attempts': 4, 'candidate_form_rows': 215, 'candidate_form_counts': {'N-Q': 99, 'N-Q/A': 0, 'N-CSR': 51, 'N-CSR/A': 7, 'N-CSRS': 51, 'N-CSRS/A': 7}, 'range_discrepancies': ['CIK0001100663-submissions-011.json', 'CIK0001100663-submissions-010.json', 'CIK0001100663-submissions-009.json', 'CIK0001100663-submissions-008.json'], 'nas_archive_sha256': 'e1194e5874e4fd46b70354e91ee47cd742421425dc81c71f617c4549e2505c4c'}


## 中文结论

四份 SEC 历史分片一次读取成功，8,037 条信托申报中固定六类表格有 215 条候选，范围 2011-03-01 至 2019-07-30。四份上级目录末日与原始分片差 1–3 天，原值均保留。原始字节与清单经 NAS 归档、远端哈希和定点恢复核验，归档 SHA-256 `e1194e5874e4fd46b70354e91ee47cd742421425dc81c71f617c4549e2505c4c`。与并行主卡核对，2011–2018 的 188 条未修订申报 accession 完全一致；另保留 13 条同期修订申报和 14 条 2019 溢出记录。0 正文、0 行情/收益、0 因子、0 拟合、0 账户。信托目录不等于 SOXX 披露篮子；正文筛选应以主卡为入口并核对修订接受时钟。
